In [1]:
import requests
import pandas as pd
from datetime import datetime

# REPLACE WITH YOUR ACTUAL KEY
API_KEY = "88c59f962d7e4c219ed3340a2ce3773b"

# Setup the endpoint
url = "https://newsapi.org/v2/everything"

In [2]:
# Configuration
QUERY = "Nvidia"
TARGET_DATE = "2026-02-09" # Example: Yesterday (Must be within last 30 days)

print(f"Configured to search for '{QUERY}' on {TARGET_DATE}...")

Configured to search for 'Nvidia' on 2026-02-09...


In [3]:
parameters = {
    "q": QUERY,
    "from": TARGET_DATE,
    "to": TARGET_DATE,      # Same day to restrict to 24h
    "sortBy": "popularity", # Options: relevancy, popularity, publishedAt
    "pageSize": 100,        # MAX allowed per request
    "language": "en",
    "apiKey": API_KEY
}

# Make the request
response = requests.get(url, params=parameters)
data = response.json()

# Basic Error Handling
if data["status"] == "ok":
    total_found = data["totalResults"]
    articles = data["articles"]
    print(f"Success! The API found {total_found} articles total.")
    print(f"Retrieved {len(articles)} articles (Free tier limit).")
else:
    print("Error:", data.get("message"))

Success! The API found 170 articles total.
Retrieved 98 articles (Free tier limit).


In [4]:
if data["status"] == "ok" and len(articles) > 0:
    # Extract only relevant fields for NLP
    clean_data = []
    for art in articles:
        clean_data.append({
            "title": art["title"],
            "description": art["description"],
            "content": art["content"],
            "source": art["source"]["name"],
            "published_at": art["publishedAt"]
        })

    # Create DataFrame
    df = pd.DataFrame(clean_data)
    
    # Remove rows where description or content is empty (crucial for training)
    df = df.dropna(subset=['description', 'content'])
    
    # Show the first few rows
    display(df.head())
    
    # Optional: Save to CSV for your SetFit project
    # df.to_csv(f"nvidia_news_{TARGET_DATE}.csv", index=False)
    # print("Saved to CSV.")
else:
    print("No data to process.")

,title,description,content,source,published_at
0,Siemens CEO Roland Busch’s mission to automate...,"Today, I’m talking with Roland Busch, who is t...",<ul><li></li><li></li><li></li></ul>\r\nRoland...,The Verge,2026-02-09T15:00:48Z
1,Nvidia Will Delay RTX 50 ‘Super’ Cards Amid St...,Nvidia delayed its planned RTX 50-series “Supe...,Nvidia delayed its planned RTX 50-series Super...,Game Revolution,2026-02-09T15:18:53Z
2,Snapdragon X2 Elite beats Apple's M5 in major ...,Benchmarks from Hardware Canucks show just how...,Qualcomm's ARM-based Snapdragon X Systems-on-C...,Windows Central,2026-02-09T14:35:29Z
3,Get a Samsung OLED gaming monitor for just $350,"To paraphrase a certain Scottish chef, “Finall...",Skip to contentWhen you purchase through links...,PCWorld,2026-02-09T18:45:50Z
4,MSI’s insane RTX 5090 Lightning will cost more...,There wasn’t much in the way of GPU announceme...,Skip to contentWhen you purchase through links...,PCWorld,2026-02-09T15:30:04Z


In [5]:
# Assuming 'articles' variable still holds the raw API data from Cell 3
if len(articles) > 0:
    clean_data = []
    for art in articles:
        clean_data.append({
            "title": art["title"],
            "description": art["description"],
            "content": art["content"],
            "source": art["source"]["name"],
            "published_at": art["publishedAt"],
            "url": art["url"]  # <--- THIS WAS MISSING
        })

    # Re-create the DataFrame
    df = pd.DataFrame(clean_data)
    
    # Remove rows where crucial data is missing
    df = df.dropna(subset=['url', 'title'])
    
    print(f"DataFrame recreated with {len(df)} rows.")
    print("Columns:", df.columns.tolist())
    
    # Check the first URL to be sure
    print(f"Sample URL: {df.iloc[0]['url']}")
else:
    print("No articles found in previous step.")

DataFrame recreated with 98 rows.
Columns: ['title', 'description', 'content', 'source', 'published_at', 'url']
Sample URL: https://www.theverge.com/podcast/875233/siemens-ceo-roland-busch-ai-automation-digital-twins-nato-tariffs


In [6]:
from newspaper import Article
import time

def get_full_article_text(url):
    try:
        article = Article(url)
        article.download()
        article.parse()
        # Optional: article.nlp() # Uncomment if you want auto-keywords/summary
        return article.text
    except Exception as e:
        # If scraping fails (403 Forbidden, timeout, etc.), return None
        return None

In [7]:
# Create a new column for full text
df['full_text'] = None

print(f"Starting scrape for {len(df)} articles...")

for index, row in df.iterrows():
    url = row['url'] # Assumes your DataFrame has a 'url' column from the API response
    
    # Fetch text
    full_text = get_full_article_text(url)
    
    # Save to DataFrame
    df.at[index, 'full_text'] = full_text
    
    # Simple progress indicator
    if index % 10 == 0:
        print(f"Processed {index} articles...")
        
    # Be polite to servers (optional but recommended)
    # time.sleep(0.5) 

# Remove rows where scraping failed (paywalls/blocks)
df_clean = df.dropna(subset=['full_text'])
df_clean = df_clean[df_clean['full_text'].str.len() > 200] # Remove very short errors

print(f"Done! Successfully scraped {len(df_clean)} full articles.")
display(df_clean[['title', 'full_text']].head())

Starting scrape for 98 articles...
Processed 0 articles...
Processed 10 articles...
Processed 20 articles...
Processed 30 articles...
Processed 40 articles...
Processed 50 articles...
Processed 60 articles...
Processed 70 articles...
Processed 80 articles...
Processed 90 articles...
Done! Successfully scraped 89 full articles.


,title,full_text
0,Siemens CEO Roland Busch’s mission to automate...,"Today, I’m talking with Roland Busch, who is t..."
1,Nvidia Will Delay RTX 50 ‘Super’ Cards Amid St...,Nvidia has delayed its highly anticipated RTX ...
2,Snapdragon X2 Elite beats Apple's M5 in major ...,Qualcomm's ARM-based Snapdragon X Systems-on-C...
3,Get a Samsung OLED gaming monitor for just $350,"To paraphrase a certain Scottish chef, “Finall..."
4,MSI’s insane RTX 5090 Lightning will cost more...,Summary created by Smart Answers AI In summary...


In [8]:
from transformers import pipeline

# Load the specific FinBERT model
# device=0 uses GPU if available, set to -1 for CPU
sentiment_pipeline = pipeline("text-classification", model="ProsusAI/finbert", return_all_scores=True)

print("FinBERT model loaded successfully.")

OSError: [WinError 1114] Proces inicializace dynamicky připojované knihovny (DLL) se nezdařil. Error loading "c:\Users\honza\Desktop\projects\stock-analysis\.venv\Lib\site-packages\torch\lib\c10.dll" or one of its dependencies.

In [ ]:
from huggingface_hub import snapshot_download

# This downloads the model files to a folder named 'local_finbert'
local_model_path = snapshot_download(repo_id="ProsusAI/finbert", local_dir="./local_finbert")

print(f"Model downloaded to: {local_model_path}")

In [ ]:
import os

# 1. Force the system to ignore OpenMP conflicts (common on Windows)
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

# 2. Import torch FIRST, before anything else
import torch

# 3. Then import transformers
from transformers import pipeline

print(f"Torch loaded successfully: {torch.__version__}")

# Now load the model
sentiment_pipeline = pipeline("text-classification", model="ProsusAI/finbert", device=-1)
print("Model loaded!")

In [9]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 98 entries, 0 to 97
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   title         98 non-null     object
 1   description   96 non-null     object
 2   content       98 non-null     object
 3   source        98 non-null     object
 4   published_at  98 non-null     object
 5   url           98 non-null     object
 6   full_text     90 non-null     object
dtypes: object(7)
memory usage: 5.5+ KB


In [11]:
df.to_excel(f"nvidia_news_{TARGET_DATE}.xlsx", index=False)